# Qmem4_sim - Can our noise-spectroscopy test be trusted? (SIMULATION ONLY)

**Everything in this notebook is SIMULATION.** No IBM hardware, no Google Drive, no API key. It says nothing about real qubits. It only answers: if a chip has slow (1/f) noise (from two-level defects, or flux noise in tunable couplers), does our pre-registered test detect it, and does it stay silent when there is none?

Simulated qubits have **known** noise: white (Markovian) dephasing, 1/f noise from telegraph fluctuators, or a mix; optionally with CONFOUNDS that are not noise but could fake a signal: static frequency offsets (+-5 kHz), static ZZ coupling (<= 1 kHz, 2 neighbours) and asymmetric readout. T1 150-400 us, 1000 shots, raw bitstrings run through the same analyse() and verdict code as the hardware notebook (Qmem4).

Required before any hardware run: **0 false positives** for M1/M2/M4 under white noise with and without confounds, and >= 19/20 detection of 1/f noise by M1 and M2. T3 is a control: the old X-only Ramsey must FAIL that requirement under frequency offsets.

In [ ]:
# T0: install (no account needed)
!pip install -q "qiskit>=1.3"
import qiskit; print('qiskit', qiskit.__version__)

In [ ]:
%%writefile noisespec.py
"""Dynamical-decoupling noise spectroscopy on every usable qubit at once.

Sequences (all qubits in parallel, waits in microseconds):
  RO0, RO1  : readout calibration (prepare |0> / |1>, measure)
  T1        : X - wait - measure                         -> P(1) decay
  RAMSEY    : H - wait - H - measure                     -> <X>
  RAMSEY_Y  : H - wait - Sdg - H - measure               -> <Y>
  ECHO      : H - w/2 - X - w/2 - H                      -> slow noise refocused
  CPMG4/16  : 4 / 16 pulses at w (k + 1/2) / n
Readout is corrected per qubit with RO0/RO1. The Ramsey coherence is the
magnitude sqrt(<X>^2 + <Y>^2), so a static frequency offset (an imperfect
drive frequency) cannot masquerade as decay. Each curve is normalised by
its shortest-wait value and fitted to C = exp(-(w/T)^beta).

Interpretation limits (stated in advance):
  * beta -> 2 only for quasi-static noise; 1/f noise gives 1 < beta < 2,
    and the T1 contribution pulls beta towards 1.
  * Simultaneous X pulses on all qubits do not refocus static ZZ coupling
    (X(x)X commutes with Z(x)Z); ZZ affects every sequence alike.
  * This measures the noise spectrum, not its origin: magnetic flux
    noise, two-level defects and charge noise can all be slow.
"""
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import \
    generate_preset_pass_manager

WAITS_US = (2, 6, 15, 35, 70, 140, 280)
SEQS = ("T1", "RAMSEY", "RAMSEY_Y", "ECHO", "CPMG4", "CPMG16")
FITTED = ("T1", "RAMSEY", "ECHO", "CPMG4", "CPMG16")
NPULSE = {"RAMSEY": 0, "RAMSEY_Y": 0, "ECHO": 1, "CPMG4": 4, "CPMG16": 16}
SHOTS = 1000


def _dt(hw, us):
    return int(us * 1e-6 / hw.dt) // 32 * 32


def circuit(hw, seq, wait_us, qubits):
    qc = QuantumCircuit(hw.num_qubits, len(qubits))
    if seq in ("RO0", "RO1"):
        if seq == "RO1":
            qc.x(qubits)
    elif seq == "T1":
        qc.x(qubits)
        qc.delay(_dt(hw, wait_us), qubits, unit="dt")
    else:
        T, n = _dt(hw, wait_us), NPULSE[seq]
        qc.h(qubits)
        if n == 0:
            qc.delay(T, qubits, unit="dt")
        else:
            seg = T // (2 * n) // 32 * 32
            qc.delay(seg, qubits, unit="dt")
            for k in range(n):
                qc.x(qubits)
                qc.delay(2 * seg if k < n - 1 else seg, qubits, unit="dt")
        if seq == "RAMSEY_Y":
            qc.sdg(qubits)
        qc.h(qubits)
    for j, q in enumerate(qubits):
        qc.measure(q, j)
    return qc


def build(hw, qubits):
    pm = generate_preset_pass_manager(
        optimization_level=0, target=hw.target,
        initial_layout=list(range(hw.num_qubits)))
    index = [("RO0", 0), ("RO1", 0)] + [(s, w) for s in SEQS
                                        for w in WAITS_US]
    return [pm.run(circuit(hw, s, w, qubits)) for s, w in index], index


def fit_decay(ws, c):
    """C = exp(-(w/T)^beta) on points with 0.05 < C < 0.95.
    Returns (T_us, beta) or (nan, nan) if fewer than 3 usable points."""
    ws, c = np.asarray(ws, float), np.asarray(c, float)
    ok = (c > 0.05) & (c < 0.95)
    if ok.sum() < 3:
        return float("nan"), float("nan")
    beta, b0 = np.polyfit(np.log(ws[ok]), np.log(-np.log(c[ok])), 1)
    T = float(np.exp(-b0 / beta)) if beta > 0 else float("nan")
    return T, float(beta)


def _p1(res, k):
    bs = res[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    return ((raw.reshape(len(bs), -1) - 48)[:, ::-1]).mean(0)


def analyse(res, index, qubits):
    """res[k].data.c bitstrings; returns {q: {seq: (T, beta)}}."""
    pos = {key: k for k, key in enumerate(index)}
    ro0 = _p1(res, pos[("RO0", 0)])               # P(1 | 0)
    ro1 = 1 - _p1(res, pos[("RO1", 0)])           # P(0 | 1)
    contrast = 1 - ro0 - ro1
    raw = {s: np.array([_p1(res, pos[(s, w)]) for w in WAITS_US])
           for s in SEQS}
    out = {}
    for j, q in enumerate(qubits):
        out[q] = {}
        if contrast[j] < 0.6:                       # unreadable qubit
            for s in FITTED:
                out[q][s] = (float("nan"), float("nan"))
            continue
        p = {s: (raw[s][:, j] - ro0[j]) / contrast[j] for s in SEQS}
        curves = {"T1": p["T1"]}
        cx, cy = 1 - 2 * p["RAMSEY"], 1 - 2 * p["RAMSEY_Y"]
        curves["RAMSEY"] = np.hypot(cx, cy)
        for s in ("ECHO", "CPMG4", "CPMG16"):
            curves[s] = 1 - 2 * p[s]
        for s in FITTED:
            v = curves[s]
            out[q][s] = ((float("nan"), float("nan")) if v[0] <= 0.2
                         else fit_decay(WAITS_US, v / v[0]))
    return out


PREDICTIONS = {
    "M1": "echo T2 > Ramsey T2* (offset-free |<X>,<Y>| magnitude): one-sided "
          "sign test p < 0.001 AND median ratio echo/Ramsey >= 1.2 "
          "(low-frequency noise present)",
    "M2": "CPMG16 T2 > echo T2: one-sided sign test p < 0.001 AND median "
          "ratio >= 1.1 (noise spectrum not white)",
    "M3": "FIT SANITY CHECK: CPMG16 T2 <= 2*T1 (+20% fit margin) on >= 95% of "
          "fitted qubits (physical bound T2 <= 2 T1)",
    "M4": "median Ramsey beta >= 1.3 (non-exponential decay => slow noise "
          "dominates dephasing) - low power for mixed noise (simulated)",
}


def _sign_p(k, n):
    """one-sided binomial P(X >= k | n, 1/2)."""
    from math import comb
    return sum(comb(n, i) for i in range(k, n + 1)) / 2 ** n


def _paired(fits, a, b):
    rows = [q for q in fits if np.isfinite(fits[q][a][0])
            and np.isfinite(fits[q][b][0]) and fits[q][b][0] > 0]
    if not rows:
        return float("nan"), float("nan"), 0
    r = np.array([fits[q][a][0] / fits[q][b][0] for q in rows])
    return _sign_p(int((r > 1).sum()), len(r)), float(np.median(r)), len(r)


def verdicts(fits):
    p1, r1, n1 = _paired(fits, "ECHO", "RAMSEY")
    p2, r2, n2 = _paired(fits, "CPMG16", "ECHO")
    rows = [q for q in fits if np.isfinite(fits[q]["CPMG16"][0])
            and np.isfinite(fits[q]["T1"][0])]
    m3 = (sum(fits[q]["CPMG16"][0] <= 2.4 * fits[q]["T1"][0] for q in rows)
          / len(rows)) if rows else float("nan")
    betas = [fits[q]["RAMSEY"][1] for q in fits
             if np.isfinite(fits[q]["RAMSEY"][1])]
    mb = float(np.median(betas)) if betas else float("nan")
    return {
        "M1": (p1 < 1e-3 and r1 >= 1.2,
               "p=%.1e, median ratio %.2f, n=%d" % (p1, r1, n1)),
        "M2": (p2 < 1e-3 and r2 >= 1.1,
               "p=%.1e, median ratio %.2f, n=%d" % (p2, r2, n2)),
        "M3": (m3 >= 0.95, "%.0f%% of %d" % (100 * m3, len(rows))),
        "M4": (mb >= 1.3, "median beta %.2f (n=%d)" % (mb, len(betas))),
    }


In [ ]:
%%writefile noisesim.py
"""Power test for noisespec: a simulated chip with KNOWN noise, run through
the same analyse() code as the hardware data (raw bitstrings in, fits out).

Per qubit:
  * T1 amplitude damping, white (Markovian) dephasing with time T_phi;
  * optional 1/f dephasing: a sum of random-telegraph fluctuators with
    log-spaced switching rates (standard model of 1/f noise);
  * optional CONFOUNDS that are not noise but could fake a signal:
      - static frequency offset d0 (imperfect drive frequency),
      - static ZZ coupling to 2 neighbours (not refocused by simultaneous
        X pulses: envelope cos(zeta t / 2)^2 on every sequence),
      - asymmetric readout (P(1|0) != P(0|1)).
A pi-pulse sequence has switching function s(t) = +-1; the accumulated
phase is phi = int s(t) (delta(t) + d0) dt, and <X> + i<Y> = env <e^{i phi}>.
"""
import numpy as np

import noisespec as N

DT = 0.5                                   # us


def _switch(seq, w, t):
    n = N.NPULSE.get(seq, 0)
    if n == 0:
        return np.ones_like(t)
    pulses = w * (np.arange(n) + 0.5) / n
    return (-1.0) ** (t[:, None] > pulses[None, :]).sum(1)


def _telegraph(rng, M, steps, rates, amp):
    out = np.zeros((M, steps))
    for g, a in zip(rates, amp, strict=True):
        flips = rng.random((M, steps)) < 1 - np.exp(-g * DT)
        s0 = rng.choice([-1.0, 1.0], size=(M, 1))
        out += a * s0 * (-1.0) ** np.cumsum(flips, axis=1)
    return out


def qubit_p1(rng, T1, Tphi, f_amp, d0=0.0, zeta=0.0, ro0=0.02, ro1=0.02,
             M=300):
    """True P(measure 1) for every (seq, wait) in noisespec's index."""
    t = np.arange(0, max(N.WAITS_US) + DT, DT)
    delta = (_telegraph(rng, M, len(t), np.logspace(-3, 0, 10),
                        f_amp * np.ones(10)) if f_amp > 0 else None)
    p = {("RO0", 0): 0.0, ("RO1", 0): 1.0}
    for s in N.SEQS:
        for w in N.WAITS_US:
            if s == "T1":
                p[(s, w)] = np.exp(-w / T1)
                continue
            k = int(round(w / DT))
            sw = _switch(s, w, t[:k])
            env = (np.exp(-w / (2 * T1)) * np.exp(-w / Tphi)
                   * np.cos(zeta * w / 2) ** 2)
            phi0 = d0 * sw.sum() * DT
            if delta is not None:
                z = np.mean(np.exp(1j * ((delta[:, :k] * sw).sum(1) * DT
                                         + phi0)))
            else:
                z = np.exp(1j * phi0)
            c = env * (z.imag if s == "RAMSEY_Y" else z.real)
            p[(s, w)] = (1 - c) / 2
    return {key: v * (1 - ro1) + (1 - v) * ro0 for key, v in p.items()}


class _Reg:
    def __init__(self, s):
        self._s = s

    def get_bitstrings(self):
        return self._s


class _Pub:
    def __init__(self, s):
        self.data = type("D", (), {"c": _Reg(s)})()


def scenario(name, n_qubits=120, seed=0, shots=N.SHOTS):
    """Simulated chip -> {q: fits} via noisespec.analyse (same code path)."""
    rng = np.random.default_rng(seed)
    base, conf = name.split("+") if "+" in name else (name, "")
    probs = []
    for _ in range(n_qubits):
        T1 = rng.uniform(150, 400)
        if base == "white":
            Tphi, fa = rng.uniform(150, 600), 0.0
        elif base == "1/f":
            Tphi, fa = 1e9, rng.lognormal(np.log(0.006), 0.4)
        elif base == "mixed":
            Tphi, fa = rng.uniform(400, 1500), rng.lognormal(np.log(0.004), 0.5)
        else:
            raise ValueError(name)
        kw = {}
        if conf == "confounds":
            kw = dict(d0=rng.uniform(-1, 1) * 2 * np.pi * 5e-3,   # +-5 kHz
                      zeta=rng.uniform(0, 1) * 2 * np.pi * 1e-3,  # <=1 kHz
                      ro0=rng.uniform(0.005, 0.02),
                      ro1=rng.uniform(0.02, 0.06))
        probs.append(qubit_p1(rng, T1, Tphi, fa, **kw))
    _, index = N.WAITS_US, [("RO0", 0), ("RO1", 0)] + [
        (s, w) for s in N.SEQS for w in N.WAITS_US]
    res = []
    for key in index:
        pr = np.array([pq[key] for pq in probs])
        bits = rng.random((shots, n_qubits)) < pr[None, :]
        res.append(_Pub(["".join("1" if b else "0" for b in row[::-1])
                         for row in bits]))
    return N.analyse(res, index, list(range(n_qubits)))


In [ ]:
# T1: one simulated chip per scenario (SIMULATION)
import importlib, numpy as np, noisespec as N, noisesim as S
importlib.reload(N); importlib.reload(S)
SCEN = ('white', 'white+confounds', '1/f', '1/f+confounds', 'mixed', 'mixed+confounds')
for name in SCEN:
    f = S.scenario(name, n_qubits=120, seed=1)
    med = {s: round(float(np.nanmedian([f[q][s][0] for q in f]))) for s in N.FITTED}
    print('==', name, '| median fitted T (us):', med)
    for k, (ok, txt) in N.verdicts(f).items():
        print('   ', k, 'HOLDS' if ok else 'FAILS', '-', txt)

In [ ]:
# T2: false-positive and detection rates over 20 independent simulated chips (SIMULATION, ~6-10 min)
import time
t0 = time.time()
RATES = {}
for name in SCEN:
    hits = {k: 0 for k in N.PREDICTIONS}
    for seed in range(20):
        for k, (ok, _) in N.verdicts(S.scenario(name, n_qubits=120, seed=100 + seed)).items():
            hits[k] += ok
    RATES[name] = hits
    print('%-16s' % name, '  '.join('%s %2d/20' % (k, v) for k, v in hits.items()), flush=True)
print('%.0f s' % (time.time() - t0))
fp = sum(RATES[n][k] for n in ('white', 'white+confounds') for k in ('M1', 'M2', 'M4'))
det = min(RATES[n][k] for n in ('1/f', '1/f+confounds') for k in ('M1', 'M2'))
print()
print('false positives, white +/- confounds (M1, M2, M4):', fp, '-> REQUIRED 0:', 'PASS' if fp == 0 else 'FAIL')
print('worst detection of 1/f +/- confounds (M1, M2): %d/20 -> REQUIRED >= 19:' % det, 'PASS' if det >= 19 else 'FAIL')

In [ ]:
# T3: CONTROL - the old X-only Ramsey (no |<X>,<Y>| magnitude) under frequency offsets (SIMULATION)
real = N.np.hypot
N.np.hypot = lambda a, b: a
try:
    fp_old = sum(N.verdicts(S.scenario('white+confounds', seed=200 + s))['M1'][0] for s in range(10))
finally:
    N.np.hypot = real
print('old X-only Ramsey, white noise + offsets: M1 false positives %d/10' % fp_old)
print('control', 'WORKS (the fix is necessary)' if fp_old > 0 else 'shows no difference')